In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project='ee-camcoredatabase')

start_date = '1990-01-01'
end_date = '2022-12-31'
bbox = ee.Geometry.BBox(-94.187, -39.020, 37.062, 18.229) 

# Load the SPEI dataset
dataset = (
    ee.ImageCollection("CSIC/SPEI/2_10")
    .filterDate(start_date, end_date)
    .filterBounds(bbox)
)

# Select the 1-month SPEI
spei01 = dataset.select('SPEI_01_month')

# Function to export each monthly image
def export_image(image):
    date = ee.Date(image.get('system:time_start')).format('YYYY-MM')
    task = ee.batch.Export.image.toDrive(
        image=image.clip(bbox),
        description=f"SPEI01_{date.getInfo()}",
        folder="GEE_SPEI_BBOX",
        scale=5000,
        region=bbox,
        maxPixels=1e13
    )
    task.start()
    print(f"Exporting: SPEI01_{date.getInfo()}")

# Iterate through each month and export
spei_list = spei01.toList(spei01.size())
for i in range(spei_list.size().getInfo()):
    image = ee.Image(spei_list.get(i))
    export_image(image)